# Final flagship notebook 03 — regime sensitivity

This notebook is the **main robustness notebook** for the regime framework. It uses a restrained sensitivity grid chosen for the paper:

- EOF components: `6, 8, 10, 12`
- regime counts: `3, 4, 5`

The notebook ends with a compact stability summary.

In [1]:
from pathlib import Path
import sys
import json
from copy import deepcopy
import pandas as pd
from IPython.display import display, Markdown

def detect_bundle_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / "src" / "flagship_predictability").exists() and (c / "examples" / "default_settings.py").exists():
            return c
    raise RuntimeError("Could not find bundle root with src/flagship_predictability and examples/default_settings.py")

BUNDLE_ROOT = detect_bundle_root()
if str(BUNDLE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT / "src"))
if str(BUNDLE_ROOT) not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT))

FLAGSHIP_OUTPUT_ROOT = (BUNDLE_ROOT / "notebooks" / "outputs" / "flagship_final") if (BUNDLE_ROOT / "notebooks").exists() else (Path.cwd() / "outputs" / "flagship_final")
FLAGSHIP_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

from examples.default_settings import SETTINGS as BASE_SETTINGS

In [2]:
from flagship_predictability import run_regime_sensitivity

SETTINGS = deepcopy(BASE_SETTINGS)
SETTINGS.blocking_threshold = 0.1
COMPONENT_GRID = [6, 8, 10, 12]
REGIME_GRID = [3, 4, 5]

display(Markdown("### Effective flagship settings"))
SETTINGS

### Effective flagship settings

AtlasConfig(truth_dataset='era5_truth_240', deterministic_models={'HRES': 'hres_0012_240', 'GraphCast': 'graphcast_2020_240', 'NeuralGCM': 'neuralgcm_det_2020_240'}, ensemble_models={'IFS-ENS': 'ifs_ens_240'}, date_windows=[('2020-01-01', '2020-03-31')], leads_hours=[24, 72, 120, 168], variables={'z500': VariableSpec(name='z500', candidates=['z', 'geopotential', 'gh'], level=500, climatology_groupby=None, thresholds=[]), 't850': VariableSpec(name='t850', candidates=['t', 'temperature'], level=850, climatology_groupby=None, thresholds=[]), 'u850': VariableSpec(name='u850', candidates=['u', 'u_component_of_wind'], level=850, climatology_groupby=None, thresholds=[]), 'v850': VariableSpec(name='v850', candidates=['v', 'v_component_of_wind'], level=850, climatology_groupby=None, thresholds=[]), 'mslp': VariableSpec(name='mslp', candidates=['msl', 'mean_sea_level_pressure', 'mslp'], level=None, climatology_groupby=None, thresholds=[])}, n_regimes=4, n_eof_components=8, bootstrap_n=400, boots

In [3]:
out = run_regime_sensitivity(
    SETTINGS,
    output_root=FLAGSHIP_OUTPUT_ROOT,
    component_grid=COMPONENT_GRID,
    regime_grid=REGIME_GRID,
)
out

{'regime_sensitivity': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/regimes/regime_sensitivity.csv')}

In [4]:
from pathlib import Path

for key, path in out.items():
    path = Path(path)
    print(f"\n--- {key} ---")
    print(path)
    if path.suffix.lower() == ".csv" and path.exists():
        try:
            display(pd.read_csv(path).head(10))
        except Exception as e:
            print("Could not preview CSV:", e)


--- regime_sensitivity ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/regimes/regime_sensitivity.csv


,window,n_components,n_regimes,regime,count,fraction,explained_variance_sum
0,2020-01-01_2020-03-31,6,3,0,107,0.293956,0.526147
1,2020-01-01_2020-03-31,6,3,1,126,0.346154,0.526147
2,2020-01-01_2020-03-31,6,3,2,131,0.359890,0.526147
3,2020-01-01_2020-03-31,6,4,0,130,0.357143,0.526147
4,2020-01-01_2020-03-31,6,4,1,107,0.293956,0.526147
5,2020-01-01_2020-03-31,6,4,2,82,0.225275,0.526147
6,2020-01-01_2020-03-31,6,4,3,45,0.123626,0.526147
7,2020-01-01_2020-03-31,6,5,0,43,0.118132,0.526147
8,2020-01-01_2020-03-31,6,5,1,75,0.206044,0.526147
9,2020-01-01_2020-03-31,6,5,2,107,0.293956,0.526147


In [5]:
df = pd.read_csv(out["regime_sensitivity"])
four = df[df["n_regimes"] == 4].copy()
pivot = four.pivot_table(index="n_components", columns="regime", values="fraction")
sorted_vals = pd.DataFrame(
    {k: sorted(row.dropna().tolist()) for k, row in pivot.iterrows()},
).T
max_spread = (sorted_vals.max(axis=0) - sorted_vals.min(axis=0)).max()
status = "PASS" if max_spread < 0.12 else ("WARN" if max_spread < 0.20 else "FAIL")

display(Markdown("### 4-regime stability summary"))
display(pivot)
display(Markdown(f"**Max spread in sorted regime fractions across EOF counts:** `{max_spread:.3f}`"))
display(Markdown(f"**Flagship regime-stability status:** {status}"))

### 4-regime stability summary

regime,0,1,2,3
n_components,,,,
6,0.357143,0.293956,0.225275,0.123626
8,0.230769,0.118132,0.354396,0.296703
10,0.118132,0.296703,0.354396,0.230769
12,0.118132,0.293956,0.362637,0.225275


**Max spread in sorted regime fractions across EOF counts:** `0.008`

**Flagship regime-stability status:** PASS